

```
https://drive.google.com/file/d/1UqjM51-pSv5gZ2TqIIFovKGmz5rE0PfG/view?usp=sharing
```



In [ ]:
# 1. Install Unsloth natively via their official pre-compiled GitHub wheel for Colab
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# 2. Install essential dependencies without dependency-resolution lockups
!pip install --no-deps xformers trl peft accelerate bitsandbytes

# 3. Install utility libraries for Kaggle integration and dataset management
!pip install kagglehub datasets transformers

In [6]:
# ==========================================
# 0. DEEP VRAM FLUSH & MEMORY PURGE
# ==========================================
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

# ==========================================
# 1. ENVIRONMENT SETUP & DATA IMPORT
# ==========================================
import kagglehub
import os
import json
from datasets import Dataset
from transformers import EarlyStoppingCallback, TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

In [7]:
# ==========================================
# 2. ROBUST DATA LOADING
# ==========================================
def load_and_format_dataset(file_path):
    formatted_data = []
    line_count = 0
    skipped_count = 0

    with open(file_path, 'r', encoding='utf-8') as f:
        first_char = f.read(1)
        f.seek(0)

        if first_char == '[':
            records = json.load(f)
        else:
            records = []
            for line in f:
                line_count += 1
                cleaned_line = line.strip()
                if not cleaned_line: continue
                try:
                    records.append(json.loads(cleaned_line))
                except json.JSONDecodeError:
                    skipped_count += 1
                    continue

    for record in records:
        jd = record.get("job_description", "")
        questions = record.get("evaluation_questions", "")
        if not jd or not questions: continue

        messages = [
            {"role": "system", "content": "You are a professional HR assistant. Analyze the provided Job Description and generate categorized resume evaluation questions."},
            {"role": "user", "content": f"Generate evaluation questions for this job description:\n\n{jd}"},
            {"role": "assistant", "content": questions}
        ]
        formatted_data.append({"messages": messages})

    print(f"Successfully loaded {len(formatted_data)} records. (Skipped: {skipped_count} rows)")
    return Dataset.from_list(formatted_data)

In [9]:
def apply_formatting_template(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

In [8]:
dataset_dir = kagglehub.dataset_download('adhamashraf202200953/techncial-generated-job-descriptions')
print(f'Data source import complete. Directory path: {dataset_dir}')

all_files = []
for root, dirs, files in os.walk(dataset_dir):
    for file in files:
        if file.endswith('.jsonl') or file.endswith('.json'):
            all_files.append(os.path.join(root, file))

INPUT_FILE = "/root/.cache/kagglehub/datasets/adhamashraf202200953/techncial-generated-job-descriptions/versions/3/phase2_questions.jsonl"
MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct"
OUTPUT_DIR = "outputs/jd_to_questions_model"
MAX_SEQ_LENGTH = 4096
LOAD_IN_4BIT = True

Using Colab cache for faster access to the 'techncial-generated-job-descriptions' dataset.
Data source import complete. Directory path: /kaggle/input/techncial-generated-job-descriptions


In [10]:
raw_dataset = load_and_format_dataset(INPUT_FILE)

dataset_split = raw_dataset.train_test_split(test_size=0.05, seed=3407)
train_dataset = dataset_split["train"]
eval_dataset = dataset_split["test"]

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

template_type = "llama-3.1" if "llama" in MODEL_NAME.lower() else ("qwen-2.5" if "qwen" in MODEL_NAME.lower() else "phi-3.5")
tokenizer = get_chat_template(tokenizer, chat_template = template_type)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


train_dataset = train_dataset.map(apply_formatting_template, batched = True)
eval_dataset = eval_dataset.map(apply_formatting_template, batched = True)

Successfully loaded 5014 records. (Skipped: 1 rows)
==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


Map:   0%|          | 0/4763 [00:00<?, ? examples/s]

Map:   0%|          | 0/251 [00:00<?, ? examples/s]

In [11]:
# ==========================================
# 4. TRAINING ARGUMENTS (THE MIDDLE GROUND)
# ==========================================
args = TrainingArguments(
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 8,
    per_device_eval_batch_size = 1,
    eval_accumulation_steps = 1,
    warmup_steps = 10,

    num_train_epochs = 1,
    learning_rate = 2e-4,
    fp16 = True,
    logging_steps = 10,

    # --- OPTIMIZED OVERFITTING CONTROLS ---
    eval_strategy = "steps",
    eval_steps = 100,
    save_strategy = "steps",
    save_steps = 100,
    save_total_limit = 2,
    load_best_model_at_end = False,
    metric_for_best_model = "eval_loss",
    greater_is_better = False,
    # ---------------------------------

    gradient_checkpointing = True,
    gradient_checkpointing_kwargs = {"use_reentrant": False},
    optim = "adamw_8bit",
    weight_decay = 0.01,
    seed = 3407,
    output_dir = "outputs",
)

```
Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
 [596/596 2:16:38, Epoch 1/1]
Step	Training Loss	Validation Loss
100	0.558434	0.540202
200	0.509068	0.515095
300	0.491300	0.503354
400	0.483224	0.494595
500	0.487584	0.489001
596	0.481372	0.486259

```

In [ ]:
trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    args = args,
    callbacks = [EarlyStoppingCallback(early_stopping_patience = 2)]
)

trainer_stats = trainer.train()

best_checkpoint_path = trainer.state.best_model_checkpoint

if best_checkpoint_path is not None:
    model.from_pretrained(model, best_checkpoint_path)
    model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
else:
    model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)

In [4]:
# Source - https://stackoverflow.com/a/52555629
# Posted by Shashank Mishra
# Retrieved 2026-05-16, License - CC BY-SA 4.0

!zip -r "llama3.2.zip" "/content/outputs"

  adding: content/outputs/ (stored 0%)
  adding: content/outputs/jd_to_questions_model/ (stored 0%)
  adding: content/outputs/jd_to_questions_model/adapter_config.json (deflated 59%)
  adding: content/outputs/jd_to_questions_model/chat_template.jinja (deflated 72%)
  adding: content/outputs/jd_to_questions_model/README.md (deflated 65%)
  adding: content/outputs/jd_to_questions_model/tokenizer_config.json (deflated 96%)
  adding: content/outputs/jd_to_questions_model/adapter_model.safetensors (deflated 72%)
  adding: content/outputs/jd_to_questions_model/tokenizer.json (deflated 85%)
  adding: content/outputs/checkpoint-500/ (stored 0%)
  adding: content/outputs/checkpoint-500/training_args.bin (deflated 53%)
  adding: content/outputs/checkpoint-500/scaler.pt (deflated 64%)
  adding: content/outputs/checkpoint-500/rng_state.pth (deflated 26%)
  adding: content/outputs/checkpoint-500/adapter_config.json (deflated 59%)
  adding: content/outputs/checkpoint-500/chat_template.jinja (deflate

In [13]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "outputs/jd_to_questions_model",
    max_seq_length = 4096,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

test_jd = """
We are looking for a Senior DevOps Engineer with 5+ years of experience.
Must have strong expertise in Kubernetes cluster administration, Terraform for AWS Infrastructure,
and building Gitlab CI/CD pipelines. Experience with Prometheus and Grafana is a plus.
"""

messages = [
    {
        "role": "system",
        "content": (
            "You are an automated HR Screening Bot. Your job is to analyze the Job Description and generate "
            "objective verification check questions for an LLM to evaluate a Candidate's CV.\n\n"
            "CRITICAL RULES:\n"
            "1. DO NOT use second-person pronouns ('you', 'your', 'yours').\n"
            "2. DO NOT frame questions as a direct interview interview ('What is your experience...').\n"
            "3. Frame questions as a third-person verification query ('Does the CV show experience with...', "
            "'Is there evidence of the candidate managing...', 'Verify if the candidate has worked on...').\n"
            "4. Maintain strict markdown headers."
        )
    },
    {
        "role": "user",
        "content": f"Generate CV evaluation verification questions for this job description:\n\n{test_jd}"
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
).to("cuda")

outputs = model.generate(
    input_ids = inputs["input_ids"],
    attention_mask = inputs["attention_mask"],
    max_new_tokens = 1024,
    use_cache = True,
    temperature = 0.5,
    top_p = 0.9
)

print(tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True))

==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
Both `max_new_tokens` (=1024) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


**Verification Questions for Senior DevOps Engineer**

**Section 1: Kubernetes Cluster Administration**
---------------------------------------------

1. **Verify if the CV shows experience with Kubernetes cluster administration**.
   Is there evidence of the candidate managing a Kubernetes cluster, including deployment, scaling, and monitoring?

2. **Verify if the candidate has experience with Kubernetes cluster configuration**.
   Does the CV show experience with configuration files such as `kubeconfig` or `kustomization`?

**Section 2: Terraform for AWS Infrastructure**
------------------------------------------

1. **Verify if the CV shows experience with Terraform for AWS Infrastructure**.
   Is there evidence of the candidate using Terraform to provision and manage AWS resources, such as EC2 instances or RDS databases?

2. **Verify if the candidate has experience with Terraform state management**.
   Does the CV show experience with managing Terraform state files and handling con